# Create Jiangxi NSF Awards (Natural Science Foundation of Jiangxi Province)

Creates awards from the Jiangxi provincial S&T department's public
省自然科学基金 拟立项公示 rosters harvested off kjt.jiangxi.gov.cn.

**Prerequisites:**
- Run `scripts/local/jiangxi_nsf_to_s3.py` first (thin runner over the shared
  `scripts/local/cn_provincial` framework; listing via the newer-Hanweb JSON
  POST `/queryList` API). It uploads
  `s3://openalex-ingest/awards/jiangxi_nsf/jiangxi_nsf_projects.parquet`.

**Data source:** https://kjt.jiangxi.gov.cn/jxskxjst/col/col27045/index.html
(通知公告 column). Public roster window: **2020 / 2022 / 2024 (main + 联合基金) /
2025** — the 公示'd categories are 重大 / 创新研究群体 / 重点 / 杰青 / 优青 /
青年直接支持 (docx attachments: 项目名称 / 项目负责人 / 申报单位 [/ 项目类别 /
立项金额]). From **2026** the roster moved behind the 科管系统 login (the
attachment line reads 需登录科管系统), so 2026+ batches are not publicly
harvestable. The general 面上/青年 categories never get a public per-project
roster (approved inside the SSO egrantweb system).

**Amounts:** only the 2024 联合基金 / 2025 重点(目标导向) rosters publish a
立项金额（万元）column (~4% of rows) -> `amount = 万元 × 10,000`, currency CNY,
NULL elsewhere. **The §6.7 amount-coverage check is waived** for the remaining
rows: provincial NSF grants carry implicit standard tiers. Documented waiver,
not a mapping miss.

**PI names:** Chinese, family-first. Per the NSFC precedent the full name is
stored in `family_name` with `given_name` NULL.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322665`
- display_name: Natural Science Foundation of Jiangxi Province
- ror_id / doi: from `openalex.common.funder` (doi 10.13039/501100004479)
- country: CN

**Priority:** `465` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jiangxi_nsf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/jiangxi_nsf/jiangxi_nsf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.jiangxi_nsf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.jiangxi_nsf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.jiangxi_nsf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322665 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.common.funder`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322665;

## Step 2: Create Natural Science Foundation of Jiangxi Province Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.jiangxi_nsf_awards
USING delta
AS
WITH
-- Path A: F4320322665 is Crossref-registered -> resolve from the dim.
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322665  -- Natural Science Foundation of Jiangxi Province
),

awards_transformed AS (
    SELECT
        -- Unique id. funder_award_id is used when the roster publishes one,
        -- else a synthetic (title + institution) key keeps ids stable across
        -- re-ingests (same shape as the Shandong pilot).
        abs(xxhash64(CONCAT(
            f.funder_id, ':',
            COALESCE(
                NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
            )
        ))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        -- Amount: 立项金额（万元） where published (~4% of rows) -> x10,000 CNY; NULL elsewhere (waived per 6.7).
        TRY_CAST(g.amount_raw AS DOUBLE) * 10000 as amount,
        CASE WHEN TRY_CAST(g.amount_raw AS DOUBLE) IS NOT NULL THEN 'CNY' ELSE CAST(NULL AS STRING) END as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from the scheme label (青年/优青/杰青/博士启动 -> fellowship;
        -- 重大/重点/群体/联合基金 -> research; else 'grant').
        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优秀青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优青%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%博士%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%重大%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            WHEN g.funder_scheme LIKE '%创新研究群体%' THEN 'research'
            WHEN g.funder_scheme LIKE '%联合基金%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'jiangxi_nsf' as provenance,

        -- Dates: only the roster year is published. start = yyyy-01-01, no end.
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN TRY_TO_DATE(CONCAT(g.start_year, '-01-01'), 'yyyy-MM-dd')
             ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,

        -- Lead investigator: Chinese PI full name in family_name, given NULL (NSFC precedent).
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE
            WHEN (g.lead_family_name IS NOT NULL AND TRIM(g.lead_family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    NULLIF(TRIM(g.lead_family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page = the announcement article the roster came from.
        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(
            f.funder_id, ':',
            COALESCE(
                NULLIF(LOWER(TRIM(g.funder_award_id)), ''),
                CONCAT(LOWER(TRIM(g.display_name)), '|', LOWER(TRIM(COALESCE(g.institution, ''))))
            )
        ))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.jiangxi_nsf_raw g
    CROSS JOIN src_funder f
    WHERE g.display_name IS NOT NULL
      AND TRIM(g.display_name) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jiangxi_nsf' AND priority = 465;

-- Insert into openalex_awards_raw with priority.
-- Priority 465: direct-from-funder ingest of Natural Science Foundation of Jiangxi Province's own
-- public rosters. Higher wins under the 2026-06-20 DESC dedup (oxjob #500),
-- so 465 outranks the acknowledgement shells (priority 0) and
-- grant-DOI stubs (priority 1).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    465 as priority
FROM openalex.awards.jiangxi_nsf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_jiangxi_nsf_awards FROM openalex.awards.jiangxi_nsf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.jiangxi_nsf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.jiangxi_nsf_awards
GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.jiangxi_nsf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.jiangxi_nsf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage. Amount: 立项金额（万元） where published (~4% of rows) -> x10,000 CNY; NULL elsewhere (waived per 6.7).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.jiangxi_nsf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'jiangxi_nsf'
GROUP BY provenance, priority;